# 02 · Pretraining (GPU T4 ×2, resumable)

**Settings:** Accelerator = GPU T4 ×2, Internet = On (only for pip/git), Persistence = Files.

**Inputs:** the output of `01_tokenizer_and_data`. **To resume in a new session:** add *this notebook's previous version output* as an extra input. The trainer searches `/kaggle/input` for the newest `ckpt_latest.pt` and continues from it (model, optimizer, GradScaler, step, data-loader position, logs). It saves and exits cleanly before `MAX_HOURS`.

In [ ]:
# --- Get the Trilingua code -------------------------------------------------------
# Option A (after the repo is public): clone it.
# Option B: upload the repo folder as a Kaggle Dataset named "trilingua-code" and attach it.
import os, sys, glob, shutil, subprocess
REPO_URL = "https://github.com/aladewealth/trilingua.git"   # TODO: confirm once the repo is pushed
CODE = "/tmp/trilingua"
attached = glob.glob("/kaggle/input/**/trilingua/model.py", recursive=True)
if os.path.exists(CODE):
    pass
elif attached:
    shutil.copytree(os.path.dirname(os.path.dirname(attached[0])), CODE)
else:
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL, CODE], check=True)
os.chdir(CODE); sys.path.insert(0, CODE)
print("code at", CODE, os.listdir(CODE))

In [ ]:
# Optional secrets (Add-ons -> Secrets). Nothing breaks if they are missing.
def get_secret(name):
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return None
for k in ("WANDB_API_KEY", "HF_TOKEN"):
    v = get_secret(k)
    if v:
        os.environ[k] = v
        print(k, "loaded")

In [ ]:
# Locate the data produced by 01_tokenizer_and_data (attach that notebook's output as input)
metas = glob.glob("/kaggle/input/**/meta.json", recursive=True)
assert metas, "Attach the output of 01_tokenizer_and_data as an input dataset"
DATA = os.path.dirname(metas[0])
print("DATA =", DATA); print(open(os.path.join(DATA, "meta.json")).read())
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print(torch.__version__, torch.cuda.device_count())

In [ ]:
OUT = '/kaggle/working/out'
MAX_HOURS = 11.0      # Kaggle sessions stop at 12 h
NGPU = torch.cuda.device_count()
BENCHMARK = False     # set True once to measure tok/s for candidate configs

## Optional: measure throughput (pick the config from measured tok/s)

In [ ]:
if BENCHMARK:
    for model, mb in [('small12', 8), ('base', 8), ('base', 16), ('base12', 8)]:
        !torchrun --standalone --nproc_per_node={NGPU} -m trilingua.train --config_file configs/pretrain_base.json \
            --data_dir {DATA} --out_dir /kaggle/working/bench --model {model} --micro_batch {mb} --benchmark_steps 10 2>&1 | grep -E 'BENCHMARK|compile|Error'
    !cat /kaggle/working/bench/benchmark.jsonl

## Train

In [ ]:
!torchrun --standalone --nproc_per_node={NGPU} -m trilingua.train --config_file configs/pretrain_base.json \
    --data_dir {DATA} --out_dir {OUT} --resume auto --resume_dirs /kaggle/input --max_hours {MAX_HOURS}

In [ ]:
from trilingua.plots import plot_losses
plot_losses(f'{OUT}/train_log.csv', f'{OUT}/val_log.csv', f'{OUT}/loss.png')
from IPython.display import Image; display(Image(f'{OUT}/loss.png'))
!tail -n 12 {OUT}/samples.txt; ls -la {OUT}